# Session 3: Deploying Models to the Cloud

**Module 5: Deep Learning in Production**  
SMU Advanced Certificate in Generative AI and Deep Learning

---

**Duration:** 4 hours (60 min theory, 180 min hands-on)  
**Prerequisites:** Basic Python knowledge, familiarity with PyTorch (from earlier modules)  
**Cloud Provider:** Amazon Web Services (AWS)

---

### What You Will Learn

By the end of this session, you will be able to:

1. Explain why cloud deployment matters for ML models
2. Understand Docker containers and why they are essential for ML
3. Navigate the AWS ecosystem for ML deployment
4. Deploy a PyTorch model to Amazon SageMaker
5. Test a live inference endpoint
6. Monitor your deployed model with CloudWatch
7. Clean up cloud resources to avoid unnecessary costs

---

### Session Outline

| Time | Section | Type |
|------|---------|------|
| 0:00 - 0:15 | Why Deploy to the Cloud? | Theory |
| 0:15 - 0:30 | Containers and Docker | Theory |
| 0:30 - 0:45 | AWS for ML Deployment | Theory |
| 0:45 - 1:00 | SageMaker Deep Dive & MLOps Basics | Theory |
| 1:00 - 1:30 | Exercise 1: Preparing a Model for Deployment | Hands-on |
| 1:30 - 2:00 | Exercise 2: Writing a Dockerfile for ML | Hands-on |
| 2:00 - 2:45 | Exercise 3: Deploying to SageMaker | Hands-on |
| 2:45 - 3:15 | Exercise 4: Testing the Endpoint | Hands-on |
| 3:15 - 3:45 | Exercise 5: Monitoring with CloudWatch | Hands-on |
| 3:45 - 4:00 | Exercise 6: Cleanup & Recap | Hands-on |

---

# Part 1: Theory (~60 minutes)

---

## 1.1 Why Deploy to the Cloud?

You have trained a deep learning model on your laptop. It works well on your test data. Now what?

To make your model useful in the real world, other people and applications need to be able to **send data to it and get predictions back**. This is what we call **deployment** — making your model available as a service.

You *could* run your model on your own computer, but that creates several problems. Cloud deployment solves them.

### Benefits of Cloud Deployment

| Benefit | What It Means | Example |
|---------|---------------|----------|
| **Scalability** | Handle 1 request or 10,000 requests per second — the cloud scales up and down automatically | An e-commerce site uses an image classifier. Traffic spikes on Black Friday, and the cloud adds more servers automatically |
| **Availability** | Your model is accessible 24/7 from anywhere in the world, with built-in redundancy | A medical imaging model needs to be available to hospitals across different time zones |
| **Managed Infrastructure** | No need to buy, maintain, or replace physical servers | You focus on improving your model, not on hardware failures or software updates |
| **GPU Access** | Use powerful GPUs on demand without purchasing expensive hardware | Run inference on an NVIDIA A10G GPU for a few hours, then stop paying for it |
| **Global Reach** | Deploy to data centres close to your users for low latency | Deploy in Singapore for Southeast Asian users, in Ireland for European users |

### Cost Considerations

Cloud services follow different pricing models. Understanding these is crucial:

| Model | How It Works | Best For |
|-------|-------------|----------|
| **Pay-per-use** | You pay only when your model is processing requests | Unpredictable or low traffic |
| **Always-on endpoint** | You pay for a running server whether it receives requests or not | Consistent, high traffic that needs low latency |
| **Spot/preemptible instances** | Discounted servers that can be interrupted | Training jobs, batch processing |

⚠️ **Important:** Cloud costs can add up quickly. A single GPU endpoint on AWS can cost **$1-$10+ per hour**. Always set up billing alerts and clean up resources you are not using. We will cover cleanup at the end of this session.

### When Cloud vs On-Premise Makes Sense

Cloud is not always the right choice. Here is a simple decision framework:

**Choose Cloud when:**
- You need to scale up or down based on demand
- You want to avoid managing hardware
- You need global availability
- You are prototyping or running short-term projects
- You do not have in-house infrastructure expertise

**Choose On-Premise when:**
- You have strict data privacy or regulatory requirements (e.g., healthcare, finance)
- You have consistent, predictable workloads that make owning hardware cheaper long-term
- You need complete control over the hardware and software stack
- You already have existing infrastructure and expertise

📝 **Note:** Many organisations use a **hybrid approach** — training models in the cloud but running inference on-premise, or vice versa. Session 4 will cover on-premise deployment in detail.

💡 **Key Concept:** Cloud deployment is not just about running your model on someone else's computer. It is about leveraging **managed services** that handle scaling, monitoring, security, and reliability so you can focus on building better models.

## 1.2 Containers and Docker

Before we deploy to the cloud, we need to understand **containers** — the technology that makes reliable deployment possible.

### What Is a Container?

Think of a **shipping container**. Before shipping containers were invented, loading cargo onto ships was chaotic — different shapes, sizes, and handling requirements. Shipping containers standardised everything: any container fits on any ship, truck, or train.

**Software containers work the same way.** A container packages your code, its dependencies, and its runtime environment into a single, standardised unit that runs the same way on any machine.

```
Without Containers:                    With Containers:
┌─────────────────────┐               ┌─────────────────────┐
│   Your Laptop       │               │   Any Machine       │
│   Python 3.9        │               │  ┌─────────────────┐│
│   PyTorch 2.0       │  "It works    │  │   Container     ││
│   CUDA 11.8         │   on my       │  │   Python 3.9    ││
│   Ubuntu 22.04      │   machine!"   │  │   PyTorch 2.0   ││
│   Special config... │               │  │   CUDA 11.8     ││
└─────────────────────┘               │  │   Your code     ││
         ↓ Deploy to server           │  │   Everything!   ││
┌─────────────────────┐               │  └─────────────────┘│
│   Server            │               │   Works exactly the ││
│   Python 3.11  ✗    │               │   same everywhere   │
│   PyTorch 1.9  ✗    │               └─────────────────────┘
│   CUDA 12.0   ✗    │
│   "It's broken!"    │
└─────────────────────┘
```

### Docker Basics

**Docker** is the most popular tool for building and running containers. Here are the key terms:

| Term | What It Is | Analogy |
|------|-----------|----------|
| **Dockerfile** | A text file with instructions to build an image | A recipe |
| **Docker Image** | A read-only template containing your app and its dependencies | A frozen meal made from the recipe |
| **Docker Container** | A running instance of an image | The meal being heated and served |
| **Docker Registry** | A place to store and share images (e.g., Docker Hub, Amazon ECR) | A supermarket freezer aisle |

### Why Containers Matter for ML

ML models are especially hard to deploy without containers because they depend on:
- Specific Python versions
- Specific library versions (PyTorch, TensorFlow, etc.)
- GPU drivers and CUDA versions
- System-level libraries (e.g., OpenCV dependencies)
- Trained model weights (often hundreds of MB or GB)

Containers capture **all of this** in a single, portable package.

### Sample Dockerfile for a PyTorch Model

Here is what a Dockerfile looks like for serving a PyTorch model with a Flask API:

```dockerfile
# Start from an official PyTorch image with GPU support
FROM pytorch/pytorch:2.1.0-cuda12.1-cudnn8-runtime

# Set the working directory inside the container
WORKDIR /app

# Copy the requirements file and install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy your model code and weights into the container
COPY model/ ./model/
COPY app.py .

# Expose the port your Flask app will run on
EXPOSE 8080

# Command to run when the container starts
CMD ["python", "app.py"]
```

We will write a complete Dockerfile in Exercise 2.

### Kubernetes (Brief Mention)

When you need to run **many containers** across **many servers**, you use **Kubernetes** (often abbreviated as K8s). Kubernetes automatically:
- Decides which server runs which container
- Restarts containers that crash
- Scales the number of containers up or down based on traffic

For this session, we will use **Amazon SageMaker**, which handles container orchestration for you behind the scenes. You do not need to learn Kubernetes to deploy with SageMaker.

💡 **Key Concept:** A Docker container packages your model, its code, and all dependencies into a single unit that runs identically on your laptop, a colleague's machine, or a cloud server. This eliminates "it works on my machine" problems.

## 1.3 AWS for ML Deployment — Overview

Amazon Web Services (AWS) is the largest cloud provider in the world. It offers dozens of services, but for ML deployment, you only need to know a handful.

### AWS Services for ML Practitioners

| Service | What It Does | How We Use It |
|---------|-------------|---------------|
| **Amazon SageMaker** | End-to-end ML platform for training, hosting, and monitoring models | Our primary deployment tool — hosts our model as an endpoint |
| **Amazon ECR** (Elastic Container Registry) | Stores Docker images in the cloud | Upload our custom Docker images so SageMaker can use them |
| **Amazon S3** (Simple Storage Service) | Object storage for any type of file | Store model weights, training data, and artifacts |
| **Amazon CloudWatch** | Monitoring, logging, and alerting | Monitor endpoint health, latency, and errors |
| **AWS Lambda** | Run code without managing servers ("serverless") | Lightweight inference for simple models (not covered in depth today) |
| **AWS IAM** (Identity and Access Management) | Control who can access what | Set permissions so SageMaker can read from S3, write logs, etc. |

### How These Services Connect

Here is the typical deployment workflow and how the services interact:

```
                        AWS Cloud
┌──────────────────────────────────────────────────────────────────┐
│                                                                  │
│   ┌──────────┐      ┌──────────────────────────────────┐        │
│   │          │      │        Amazon SageMaker           │        │
│   │ Amazon   │      │                                    │        │
│   │   S3     │─────▶│  ┌────────┐    ┌──────────────┐  │        │
│   │          │      │  │ Model  │───▶│   Endpoint    │  │        │
│   │ model.   │      │  │ Object │    │  (Runs your   │  │        │
│   │ tar.gz   │      │  └────────┘    │   container)  │  │        │
│   │          │      │                 └───────┬───────┘  │        │
│   └──────────┘      │                         │          │        │
│                      └─────────────────────────┼──────────┘        │
│                                                │                   │
│   ┌──────────┐                                │                   │
│   │ Amazon   │                                │                   │
│   │  ECR     │── Docker Image ────────────────┘                   │
│   │          │   (inference code)                                  │
│   └──────────┘                                                    │
│                      ┌──────────────┐                             │
│                      │  CloudWatch  │◀── Logs & Metrics ──────── │
│                      │  (Monitoring)│                             │
│                      └──────────────┘                             │
│                                                                   │
└──────────────────────────────────────────────────────────────────┘
         ▲                                      │
         │           HTTPS Request              │
         │         ┌──────────────┐             │
         └─────────│  Your App /  │◀────────────┘
                   │  Client      │   Prediction Response
                   └──────────────┘
```

**The flow is:**
1. You upload your model artifacts (weights) to **S3**
2. Your Docker image (inference code) is stored in **ECR** (or you use a built-in SageMaker container)
3. **SageMaker** pulls the model from S3 and the container from ECR, and launches an **endpoint**
4. Your application sends HTTPS requests to the endpoint and receives predictions
5. **CloudWatch** collects logs and metrics from the endpoint for monitoring

📝 **Note:** For this session, we will use **SageMaker's built-in PyTorch container** so we do not need to build our own Docker image from scratch or use ECR. This is the simplest path for beginners.

## 1.4 Amazon SageMaker Deep Dive

Amazon SageMaker is AWS's fully managed ML platform. It covers the entire ML lifecycle, but today we focus on **model hosting** (deployment).

### What SageMaker Offers

| Feature | Description |
|---------|-------------|
| **SageMaker Notebooks** | Managed Jupyter notebooks with pre-installed ML libraries |
| **Training Jobs** | Managed training on scalable infrastructure (not covered today) |
| **Model Hosting** | Deploy models as real-time or batch endpoints |
| **Model Monitor** | Automatically detect data drift and model quality issues |
| **Pipelines** | Build automated ML workflows |

### Real-Time vs Batch Inference

SageMaker supports two main types of inference:

| Type | How It Works | Latency | Cost | Use Case |
|------|-------------|---------|------|----------|
| **Real-Time Endpoint** | Always-on server waiting for requests | Milliseconds | Higher (pay while running) | Interactive apps, APIs, chatbots |
| **Batch Transform** | Process a large dataset all at once | Minutes to hours | Lower (pay per job) | Scoring a million records overnight |
| **Serverless Inference** | Scales to zero when idle | Seconds (cold start) | Lowest for sporadic traffic | Infrequent or unpredictable traffic |

Today we will deploy a **real-time endpoint** — the most common deployment pattern.

### SageMaker Model Hosting Workflow

Deploying a model to SageMaker involves five steps:

```
Step 1          Step 2           Step 3              Step 4           Step 5
┌─────────┐    ┌──────────┐    ┌───────────────┐    ┌────────────┐   ┌─────────────┐
│ Upload  │    │ Create   │    │ Create        │    │ Deploy to  │   │ Send        │
│ model   │───▶│ SageMaker│───▶│ Endpoint      │───▶│ an         │──▶│ inference   │
│ to S3   │    │ Model    │    │ Configuration │    │ Endpoint   │   │ requests    │
└─────────┘    └──────────┘    └───────────────┘    └────────────┘   └─────────────┘
                                                     (~5-10 min)
```

**Step 1: Upload model artifacts to S3**  
SageMaker expects your model files packaged as a `model.tar.gz` file stored in an S3 bucket.

**Step 2: Create a SageMaker Model**  
This tells SageMaker where to find your model artifacts (S3 path) and which Docker container to use for inference.

**Step 3: Create an Endpoint Configuration**  
This specifies the instance type (e.g., `ml.m5.large`), the number of instances, and which model to use.

**Step 4: Deploy to an Endpoint**  
SageMaker provisions the compute resources, downloads your model, starts your container, and begins accepting requests. This typically takes **5-10 minutes**.

**Step 5: Send inference requests**  
Your application sends data to the endpoint's URL and receives predictions back.

### Pricing Basics

SageMaker endpoint pricing depends on:
- **Instance type**: `ml.t2.medium` (~$0.05/hr) vs `ml.p3.2xlarge` (~$3.82/hr)
- **Number of instances**: More instances = higher cost but more capacity
- **Duration**: You pay for every second the endpoint is running

⚠️ **Important:** For this hands-on session, we will use `ml.m5.large` (approximately $0.12/hour). **Remember to delete your endpoint at the end of the session!** An endpoint left running for a month would cost approximately $86.

💡 **Key Concept:** SageMaker abstracts away the complexity of container orchestration, load balancing, and auto-scaling. You provide a model and a container, and SageMaker handles the rest.

## 1.5 MLOps Basics

### What Is MLOps?

**MLOps** (Machine Learning Operations) applies **DevOps principles** to ML systems. Just as DevOps automates software delivery, MLOps automates the ML lifecycle — from training to deployment to monitoring.

In traditional software, once you deploy, the code does not change by itself. In ML, the world changes around your model:
- User behaviour shifts
- Data distributions evolve
- New categories of input appear

MLOps helps you **detect** and **respond** to these changes.

### Monitoring Model Performance in Production

Once your model is deployed, you need to watch for two types of issues:

| What to Monitor | Why | How |
|-----------------|-----|-----|
| **Infrastructure metrics** | Is the endpoint healthy and responsive? | CloudWatch: CPU, memory, latency, error rates |
| **Model quality metrics** | Is the model still making good predictions? | SageMaker Model Monitor: accuracy, precision, recall |
| **Data quality metrics** | Is the input data still similar to training data? | SageMaker Model Monitor: feature distributions, missing values |

### Model Drift

**Model drift** (also called **concept drift**) happens when the relationship between input features and the target variable changes over time.

**Example:** You trained a sentiment classifier on movie reviews from 2020. By 2024, language patterns have shifted — new slang, new topics — and your model's accuracy drops.

**When to retrain:**
- Model accuracy drops below a threshold
- Input data distribution changes significantly
- Business requirements change (new classes, new objectives)
- On a regular schedule (e.g., monthly) as a precaution

### CloudWatch for Logging and Alerts

Amazon **CloudWatch** is AWS's monitoring service. For ML endpoints, it tracks:
- **Invocation count**: How many requests your endpoint received
- **Latency**: How long each request takes (ModelLatency, OverheadLatency)
- **Error rates**: 4xx (client errors) and 5xx (server errors)
- **CPU/Memory utilisation**: How hard your instance is working

You can create **alarms** that notify you when metrics exceed thresholds — for example, if error rate goes above 5% or latency exceeds 1 second.

### SageMaker Model Monitor (Overview)

SageMaker Model Monitor goes beyond infrastructure monitoring. It can:
1. Capture input/output data from your endpoint
2. Compare incoming data distributions to your training data
3. Detect data quality issues (missing values, type mismatches)
4. Alert you when drift is detected

Setting up Model Monitor in full is beyond today's scope, but we will cover CloudWatch monitoring in Exercise 5.

💡 **Key Concept:** Deploying a model is not the end — it is the beginning of a new phase. You must continuously monitor your model's health and performance, and be ready to retrain when drift is detected.

---

# Part 2: Hands-on Exercises (~180 minutes)

---

⚠️ **Important: AWS Account Required**

The following exercises require an AWS account with:
- An **IAM role** with SageMaker, S3, and CloudWatch permissions
- An **S3 bucket** for storing model artifacts
- **AWS credentials** configured on your machine (via `aws configure` or environment variables)

If you are using AWS Academy or a provided lab environment, your instructor will give you these details.

📝 **Note:** Look for placeholders like `YOUR_S3_BUCKET` and `YOUR_ROLE_ARN` — replace these with your actual AWS details.

### Install Required Packages

Run the cell below to install all the packages needed for this session. If you are running on SageMaker Studio or an AWS notebook instance, most of these are pre-installed.

In [1]:
# Install required packages
# boto3: AWS SDK for Python
# sagemaker: High-level SageMaker SDK
# torch & torchvision: PyTorch and its vision utilities
# flask: Lightweight web server (for Exercise 2)
# requests: HTTP library for testing endpoints

!pip install boto3 sagemaker torch torchvision flask requests --quiet

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
# Verify installations and print versions
import boto3
import sagemaker
import torch
import torchvision

print(f"boto3 version:       {boto3.__version__}")
print(f"sagemaker version:   {sagemaker.__version__}")
print(f"PyTorch version:     {torch.__version__}")
print(f"torchvision version: {torchvision.__version__}")
print(f"CUDA available:      {torch.cuda.is_available()}")


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/xdg-ubuntu/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/user/.config/sagemaker/config.yaml


boto3 version:       1.42.63
sagemaker version:   2.257.1
PyTorch version:     2.10.0+cu128
torchvision version: 0.25.0+cu128
CUDA available:      True


### Configure AWS Settings

Fill in your AWS details below. These will be used throughout the exercises.

⚠️ **Important:** Never commit your AWS credentials to version control. The cell below uses IAM roles (recommended) or environment variables — not hardcoded keys.

In [3]:
import boto3
import sagemaker
import os

# ============================================================
# CONFIGURATION
# ============================================================

# Set the AWS profile for SSO authentication
os.environ["AWS_PROFILE"] = "sso-holt"

# SageMaker execution role
role = "arn:aws:iam::359143808088:role/SageMakerExecutionRole-Module6"

# S3 bucket for storing model artifacts
bucket_name = "sagemaker-ap-southeast-1-359143808088"

# AWS region
region = "ap-southeast-1"

# Create a SageMaker session
boto_session = boto3.Session(region_name=region, profile_name="sso-holt")
sagemaker_session = sagemaker.Session(boto_session=boto_session)

print(f"IAM role:            {role}")
print(f"S3 bucket:           {bucket_name}")
print(f"AWS region:          {region}")
print(f"SageMaker session:   Ready")


IAM role:            arn:aws:iam::359143808088:role/SageMakerExecutionRole-Module6
S3 bucket:           sagemaker-ap-southeast-1-359143808088
AWS region:          ap-southeast-1
SageMaker session:   Ready


---

## 🔧 Exercise 1: Preparing a Model for Deployment (~30 minutes)

In this exercise, we will:
1. Load a pre-trained image classification model (ResNet-18 from torchvision)
2. Save it in a deployment-ready format
3. Package it as `model.tar.gz` (SageMaker's expected format)
4. Upload the package to Amazon S3

💡 **Key Concept:** SageMaker expects model artifacts in a specific format — a `.tar.gz` archive containing your model files. The exact file structure depends on the framework you are using.

### Step 1.1: Load a Pre-trained Model

We will use **ResNet-18**, a well-known image classification model pre-trained on ImageNet (1000 classes). This gives us a real, usable model without needing to train anything.

**Expected output:** The model architecture summary and confirmation that weights are loaded.

In [4]:
import torch
import torchvision.models as models

# Load the pre-trained ResNet-18 model
# 'weights=IMAGENET1K_V1' loads weights trained on ImageNet
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Set the model to evaluation mode
# This disables dropout and batch normalization training behaviour
model.eval()

print("Model loaded successfully!")
print(f"Model type: {type(model).__name__}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Output classes: {model.fc.out_features}")

Model loaded successfully!
Model type: ResNet
Number of parameters: 11,689,512
Output classes: 1000


### Step 1.2: Test the Model Locally

Before deploying, let us verify the model works by running a quick inference with a random input tensor.

**Expected output:** A tensor of shape `[1, 1000]` representing probabilities for 1000 ImageNet classes.

In [5]:
import torch

# Create a random input tensor with the shape ResNet expects:
# batch_size=1, channels=3 (RGB), height=224, width=224
dummy_input = torch.randn(1, 3, 224, 224)

# Run inference (no gradient computation needed)
with torch.no_grad():
    output = model(dummy_input)

print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Output type:  {output.dtype}")

# Get the predicted class
predicted_class = torch.argmax(output, dim=1).item()
print(f"Predicted class index: {predicted_class}")
print("\nModel is working correctly!")

Input shape:  torch.Size([1, 3, 224, 224])
Output shape: torch.Size([1, 1000])
Output type:  torch.float32
Predicted class index: 107

Model is working correctly!


### Step 1.3: Save the Model in Deployment-Ready Format

For SageMaker's built-in PyTorch container, we need to save the model using `torch.save()` and create an **inference script** (`inference.py`) that tells SageMaker how to:
- Load the model (`model_fn`)
- Process input data (`input_fn`)
- Run the model (`predict_fn`)
- Format the output (`output_fn`)

📝 **Note:** The SageMaker PyTorch container looks for a file called `model.pth` for the model weights and `inference.py` (inside a `code/` directory) for the inference logic.

In [6]:
import os

# Create a directory to hold our model artifacts
model_dir = "model_artifacts"
code_dir = os.path.join(model_dir, "code")
os.makedirs(code_dir, exist_ok=True)

# Save the model weights
model_path = os.path.join(model_dir, "model.pth")
torch.save(model.state_dict(), model_path)

print(f"Model saved to: {model_path}")
print(f"Model file size: {os.path.getsize(model_path) / (1024*1024):.1f} MB")

Model saved to: model_artifacts/model.pth
Model file size: 44.7 MB


In [7]:
# Create the inference script that SageMaker will use
# This script defines how SageMaker loads and uses your model

inference_script = '''
import os
import json
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import io
import numpy as np


def model_fn(model_dir):
    """
    Load the PyTorch model from the model directory.
    SageMaker calls this function when the endpoint starts.
    
    Args:
        model_dir: Directory where model artifacts are stored
    
    Returns:
        The loaded PyTorch model
    """
    model = models.resnet18()
    model_path = os.path.join(model_dir, "model.pth")
    model.load_state_dict(torch.load(model_path, map_location=torch.device("cpu")))
    model.eval()
    return model


def input_fn(request_body, request_content_type):
    """
    Deserialize and preprocess the input data.
    SageMaker calls this for each inference request.
    
    Args:
        request_body: The raw request body (bytes)
        request_content_type: The content type of the request
    
    Returns:
        Preprocessed tensor ready for the model
    """
    if request_content_type == "application/json":
        # Accept JSON with a list of pixel values
        data = json.loads(request_body)
        tensor = torch.tensor(data["inputs"], dtype=torch.float32)
        return tensor
    elif request_content_type in ["image/jpeg", "image/png"]:
        # Accept image files directly
        image = Image.open(io.BytesIO(request_body)).convert("RGB")
        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
        ])
        tensor = transform(image).unsqueeze(0)  # Add batch dimension
        return tensor
    else:
        raise ValueError(f"Unsupported content type: {request_content_type}")


def predict_fn(input_data, model):
    """
    Run the model on the preprocessed input.
    
    Args:
        input_data: The preprocessed input tensor
        model: The loaded PyTorch model
    
    Returns:
        Model predictions
    """
    with torch.no_grad():
        output = model(input_data)
        probabilities = torch.nn.functional.softmax(output, dim=1)
    return probabilities


def output_fn(prediction, response_content_type):
    """
    Serialize the model output to send back to the client.
    
    Args:
        prediction: The model output tensor
        response_content_type: The desired response format
    
    Returns:
        Serialized prediction
    """
    # Get top 5 predictions
    top5_prob, top5_idx = torch.topk(prediction, 5, dim=1)
    
    result = {
        "predictions": [
            {
                "class_index": idx.item(),
                "probability": prob.item()
            }
            for idx, prob in zip(top5_idx[0], top5_prob[0])
        ]
    }
    return json.dumps(result)
'''

# Write the inference script to the code directory
inference_path = os.path.join(code_dir, "inference.py")
with open(inference_path, "w") as f:
    f.write(inference_script)

print(f"Inference script saved to: {inference_path}")
print("\nDirectory structure:")
for root, dirs, files in os.walk(model_dir):
    level = root.replace(model_dir, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

Inference script saved to: model_artifacts/code/inference.py

Directory structure:
model_artifacts/
  model.pth
  code/
    inference.py


### Step 1.4: Package as model.tar.gz

SageMaker expects model artifacts to be packaged as a `.tar.gz` file with a specific structure:

```
model.tar.gz
├── model.pth          # Model weights
└── code/
    └── inference.py   # Inference script
```

**Expected output:** A compressed archive file ready for upload.

In [8]:
import tarfile

# Create the tar.gz archive
tar_path = "model.tar.gz"

with tarfile.open(tar_path, "w:gz") as tar:
    # Add model weights
    tar.add(os.path.join(model_dir, "model.pth"), arcname="model.pth")
    # Add inference script in the code/ directory
    tar.add(os.path.join(code_dir, "inference.py"), arcname="code/inference.py")

print(f"Archive created: {tar_path}")
print(f"Archive size: {os.path.getsize(tar_path) / (1024*1024):.1f} MB")

# Verify the contents of the archive
print("\nArchive contents:")
with tarfile.open(tar_path, "r:gz") as tar:
    for member in tar.getmembers():
        print(f"  {member.name} ({member.size / (1024*1024):.1f} MB)")

Archive created: model.tar.gz
Archive size: 41.4 MB

Archive contents:
  model.pth (44.7 MB)
  code/inference.py (0.0 MB)


### Step 1.5: Upload to Amazon S3

Now we upload our model archive to S3 so SageMaker can access it.

**Expected output:** An S3 URI like `s3://your-bucket/models/resnet18/model.tar.gz`

⚠️ **Important:** Make sure you have replaced `YOUR_S3_BUCKET` in the configuration cell above with your actual bucket name.

In [9]:
import boto3

# Create an S3 client
s3_client = boto3.client("s3", region_name=region)

# Define the S3 key (path within the bucket)
s3_key = "models/resnet18/model.tar.gz"

# Upload the model archive to S3
print(f"Uploading {tar_path} to s3://{bucket_name}/{s3_key} ...")
s3_client.upload_file(tar_path, bucket_name, s3_key)

# Construct the full S3 URI
model_s3_uri = f"s3://{bucket_name}/{s3_key}"
print(f"\nUpload complete!")
print(f"Model S3 URI: {model_s3_uri}")

# Verify the upload by listing the file
response = s3_client.head_object(Bucket=bucket_name, Key=s3_key)
print(f"File size in S3: {response['ContentLength'] / (1024*1024):.1f} MB")
print(f"Last modified: {response['LastModified']}")

Uploading model.tar.gz to s3://sagemaker-ap-southeast-1-359143808088/models/resnet18/model.tar.gz ...



Upload complete!
Model S3 URI: s3://sagemaker-ap-southeast-1-359143808088/models/resnet18/model.tar.gz
File size in S3: 41.4 MB
Last modified: 2026-04-08 04:28:42+00:00


📝 **Note:** You have now completed the first step of the SageMaker deployment workflow — your model artifacts are in S3 and ready to be deployed.

---

## 🔧 Exercise 2: Writing a Dockerfile for ML (~30 minutes)

In this exercise, we will write a Dockerfile that packages our PyTorch model with a Flask inference server. This shows you what happens "under the hood" when SageMaker runs your model in a container.

📝 **Note:** We will **not** actually build and push this Docker image during the session (that requires Docker installed locally). Instead, we will write and understand the Dockerfile. In Exercise 3, we will use SageMaker's **built-in container**, which does this for you.

💡 **Key Concept:** Understanding Dockerfiles is important even when using managed services like SageMaker. It helps you debug issues and customize behaviour when the built-in containers do not meet your needs.

### Step 2.1: The Flask Inference Server

First, let us write a simple Flask application that serves our model. This is the kind of code that runs inside the Docker container.

SageMaker containers follow a specific contract:
- **`/ping`** — Health check endpoint (returns 200 if the model is loaded)
- **`/invocations`** — Prediction endpoint (receives data, returns predictions)

In [10]:
# This cell writes a Flask inference server to a file
# We are NOT running it here — just creating the file for our Docker image

flask_app_code = '''
import os
import json
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from flask import Flask, request, jsonify
from PIL import Image
import io

# Initialize the Flask application
app = Flask(__name__)

# Global variable for our model
model = None

# Image preprocessing pipeline — must match what the model was trained with
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


def load_model():
    """Load the PyTorch model from disk."""
    global model
    model_path = os.environ.get("MODEL_PATH", "/opt/ml/model/model.pth")
    model = models.resnet18()
    model.load_state_dict(torch.load(model_path, map_location="cpu"))
    model.eval()
    print(f"Model loaded from {model_path}")


@app.route("/ping", methods=["GET"])
def ping():
    """
    Health check endpoint.
    SageMaker calls this to verify the container is ready.
    Returns 200 if the model is loaded, 500 otherwise.
    """
    if model is not None:
        return jsonify({"status": "healthy"}), 200
    else:
        return jsonify({"status": "not ready"}), 500


@app.route("/invocations", methods=["POST"])
def predict():
    """
    Prediction endpoint.
    Accepts an image and returns top-5 predicted ImageNet classes.
    """
    try:
        # Read the image from the request
        if request.content_type in ["image/jpeg", "image/png"]:
            image = Image.open(io.BytesIO(request.data)).convert("RGB")
        else:
            return jsonify({"error": f"Unsupported content type: {request.content_type}"}), 400

        # Preprocess the image
        input_tensor = transform(image).unsqueeze(0)

        # Run inference
        with torch.no_grad():
            output = model(input_tensor)
            probabilities = torch.nn.functional.softmax(output, dim=1)

        # Get top-5 predictions
        top5_prob, top5_idx = torch.topk(probabilities, 5, dim=1)
        predictions = [
            {"class_index": idx.item(), "probability": round(prob.item(), 4)}
            for idx, prob in zip(top5_idx[0], top5_prob[0])
        ]

        return jsonify({"predictions": predictions})

    except Exception as e:
        return jsonify({"error": str(e)}), 500


if __name__ == "__main__":
    load_model()
    # SageMaker sends traffic to port 8080
    app.run(host="0.0.0.0", port=8080)
'''

# Save the Flask app to a file
os.makedirs("docker_example", exist_ok=True)
with open("docker_example/app.py", "w") as f:
    f.write(flask_app_code)

print("Flask inference server written to: docker_example/app.py")
print("\nThis server provides two endpoints:")
print("  GET  /ping         - Health check")
print("  POST /invocations  - Inference")

Flask inference server written to: docker_example/app.py

This server provides two endpoints:
  GET  /ping         - Health check
  POST /invocations  - Inference


### Step 2.2: Write the Dockerfile

Now let us write the Dockerfile. Each line is explained with comments.

In [11]:
# Write a Dockerfile for our ML inference server

dockerfile_content = '''
# ==============================================================
# Dockerfile for PyTorch Model Inference Server
# ==============================================================

# LINE 1: Start from the official PyTorch image
# This base image includes Python, PyTorch, and CUDA support.
# Using the "runtime" version (smaller than the full "devel" image)
FROM pytorch/pytorch:2.1.0-cuda12.1-cudnn8-runtime

# LINE 2: Set the working directory inside the container
# All subsequent commands will run from /app
WORKDIR /app

# LINE 3: Install system dependencies
# libgl1-mesa-glx is needed for OpenCV (image processing)
RUN apt-get update && apt-get install -y --no-install-recommends \\
    libgl1-mesa-glx \\
    libglib2.0-0 \\
    && rm -rf /var/lib/apt/lists/*

# LINE 4: Copy and install Python dependencies
# We copy requirements.txt first to leverage Docker's layer caching.
# If requirements.txt has not changed, Docker skips this step on rebuild.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# LINE 5: Copy the model weights
# In production, model weights are often downloaded from S3 at startup.
# Here we bake them into the image for simplicity.
COPY model.pth /opt/ml/model/model.pth

# LINE 6: Copy the Flask inference server
COPY app.py .

# LINE 7: Set environment variables
# Tell Python to not buffer output (so logs appear immediately)
ENV PYTHONUNBUFFERED=1
ENV MODEL_PATH=/opt/ml/model/model.pth

# LINE 8: Expose port 8080
# This is the port SageMaker expects for inference traffic
EXPOSE 8080

# LINE 9: Health check
# Docker will periodically call /ping to check if the container is healthy
HEALTHCHECK --interval=30s --timeout=5s --retries=3 \\
    CMD curl -f http://localhost:8080/ping || exit 1

# LINE 10: Run the Flask server when the container starts
CMD ["python", "app.py"]
'''

# Save the Dockerfile
with open("docker_example/Dockerfile", "w") as f:
    f.write(dockerfile_content)

print("Dockerfile written to: docker_example/Dockerfile")
print("\n" + "=" * 60)
print("Dockerfile Contents (with explanations):")
print("=" * 60)
print(dockerfile_content)

Dockerfile written to: docker_example/Dockerfile

Dockerfile Contents (with explanations):

# ==============================================================
# Dockerfile for PyTorch Model Inference Server
# ==============================================================

# LINE 1: Start from the official PyTorch image
# This base image includes Python, PyTorch, and CUDA support.
# Using the "runtime" version (smaller than the full "devel" image)
FROM pytorch/pytorch:2.1.0-cuda12.1-cudnn8-runtime

# LINE 2: Set the working directory inside the container
# All subsequent commands will run from /app
WORKDIR /app

# LINE 3: Install system dependencies
# libgl1-mesa-glx is needed for OpenCV (image processing)
RUN apt-get update && apt-get install -y --no-install-recommends \
    libgl1-mesa-glx \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

# LINE 4: Copy and install Python dependencies
# We copy requirements.txt first to leverage Docker's layer caching.
# If requirements.txt has n

In [12]:
# Create a requirements.txt for the Docker image

requirements_content = """flask==3.0.0
Pillow==10.1.0
torchvision==0.16.0
gunicorn==21.2.0
"""

with open("docker_example/requirements.txt", "w") as f:
    f.write(requirements_content)

print("requirements.txt written to: docker_example/requirements.txt")
print("\nContents:")
print(requirements_content)

requirements.txt written to: docker_example/requirements.txt

Contents:
flask==3.0.0
Pillow==10.1.0
torchvision==0.16.0
gunicorn==21.2.0



### Step 2.3: How to Build and Test Locally

If you have Docker installed on your machine, here are the commands to build and test this container. **You do not need to run these now** — they are for reference.

#### Build the Docker Image

```bash
# Navigate to the docker_example directory
cd docker_example

# Copy the model weights into this directory
cp ../model_artifacts/model.pth .

# Build the Docker image
# -t gives the image a name ("tag")
# . tells Docker to use the current directory as the build context
docker build -t resnet18-inference .
```

#### Run the Container Locally

```bash
# Run the container
# -p maps port 8080 on your machine to port 8080 in the container
# -d runs the container in the background
docker run -p 8080:8080 -d resnet18-inference
```

#### Test with a Request

```bash
# Health check
curl http://localhost:8080/ping

# Send an image for prediction
curl -X POST http://localhost:8080/invocations \
     -H "Content-Type: image/jpeg" \
     --data-binary @test_image.jpg
```

#### Push to Amazon ECR

To use your custom container with SageMaker, you push it to **Amazon ECR** (Elastic Container Registry):

```bash
# 1. Create an ECR repository
aws ecr create-repository --repository-name resnet18-inference

# 2. Authenticate Docker to ECR
aws ecr get-login-password --region ap-southeast-1 | \
    docker login --username AWS --password-stdin \
    123456789012.dkr.ecr.ap-southeast-1.amazonaws.com

# 3. Tag the image with the ECR repository URI
docker tag resnet18-inference:latest \
    123456789012.dkr.ecr.ap-southeast-1.amazonaws.com/resnet18-inference:latest

# 4. Push the image to ECR
docker push \
    123456789012.dkr.ecr.ap-southeast-1.amazonaws.com/resnet18-inference:latest
```

📝 **Note:** For this session, we skip building and pushing a custom container. In Exercise 3, we will use SageMaker's **built-in PyTorch container**, which already includes PyTorch, a model serving framework, and handles all the container setup for you.

---

## 🔧 Exercise 3: Deploying to SageMaker (Using Built-in Container) (~45 minutes)

Now we deploy our model to a live endpoint using the SageMaker Python SDK. We will use a **SageMaker built-in PyTorch container** — the simplest path for beginners.

⚠️ **Important:** This exercise creates a **real AWS resource** that costs money. The `ml.m5.large` instance costs approximately **$0.12/hour**. Make sure to complete Exercise 6 (Cleanup) to delete the endpoint when you are done.

📝 **Note:** Endpoint creation typically takes **5-10 minutes**. SageMaker needs to provision compute resources, download the container image, download your model from S3, and start the inference server.

### Required IAM Permissions

Your SageMaker execution role needs the following permissions:
- `sagemaker:CreateModel`, `sagemaker:CreateEndpointConfig`, `sagemaker:CreateEndpoint`
- `s3:GetObject` (to download model artifacts)
- `ecr:GetAuthorizationToken`, `ecr:BatchGetImage` (to pull the container image)
- `logs:CreateLogGroup`, `logs:CreateLogStream`, `logs:PutLogEvents` (for CloudWatch logging)

If you are using the `AmazonSageMakerFullAccess` managed policy, these permissions are already included.

### Step 3.1: Create a SageMaker PyTorch Model

The SageMaker SDK provides a `PyTorchModel` class that wraps all the complexity of creating a model object. It automatically selects the right built-in container for your PyTorch version.

**Expected output:** A SageMaker Model object with a reference to your S3 model artifacts.

In [13]:
from sagemaker.pytorch import PyTorchModel

# Create a SageMaker PyTorch Model
# This tells SageMaker:
#   - Where to find the model artifacts (S3)
#   - Which IAM role to use
#   - Which PyTorch version to use (determines the container image)
#   - Which Python version to use

pytorch_model = PyTorchModel(
    model_data=model_s3_uri,             # S3 path to model.tar.gz
    role=role,                            # IAM role ARN
    framework_version="2.1.0",            # PyTorch version
    py_version="py310",                   # Python version
    entry_point="inference.py",           # Entry point script (inside model.tar.gz/code/)
    sagemaker_session=sagemaker_session,
    name="resnet18-image-classifier",     # A friendly name for the model
)

print("SageMaker PyTorch Model created!")
print(f"  Model data:       {model_s3_uri}")
print(f"  Framework:        PyTorch 2.1.0")
print(f"  Python version:   3.10")
print(f"  Entry point:      inference.py")

SageMaker PyTorch Model created!
  Model data:       s3://sagemaker-ap-southeast-1-359143808088/models/resnet18/model.tar.gz
  Framework:        PyTorch 2.1.0
  Python version:   3.10
  Entry point:      inference.py


### Step 3.2: Deploy to an Endpoint

The `deploy()` method handles Steps 2-4 of the SageMaker workflow:
1. Creates the SageMaker Model
2. Creates an Endpoint Configuration
3. Creates and starts the Endpoint

⚠️ **Important:** This step will take approximately **5-10 minutes**. The cell will show progress indicators. Do not interrupt it.

**Expected output:** A SageMaker Predictor object connected to the live endpoint.

In [14]:
import time

# Define the endpoint name
# Using a timestamp to make the name unique
endpoint_name = f"resnet18-endpoint-{int(time.time())}"

print(f"Deploying endpoint: {endpoint_name}")
print(f"Instance type: ml.m5.large (~$0.12/hour)")
print(f"Instance count: 1")
print(f"\nThis will take approximately 5-10 minutes...")
print(f"Started at: {time.strftime('%H:%M:%S')}")
print()

# Deploy the model to an endpoint
predictor = pytorch_model.deploy(
    instance_type="ml.m5.large",     # Compute instance type
    initial_instance_count=1,         # Number of instances
    endpoint_name=endpoint_name,      # Custom endpoint name
)

print(f"\nEndpoint deployed successfully!")
print(f"Endpoint name: {endpoint_name}")
print(f"Finished at: {time.strftime('%H:%M:%S')}")

Deploying endpoint: resnet18-endpoint-1775622522
Instance type: ml.m5.large (~$0.12/hour)
Instance count: 1

This will take approximately 5-10 minutes...
Started at: 12:28:42



-

-

-

-

-

-

!


Endpoint deployed successfully!
Endpoint name: resnet18-endpoint-1775622522
Finished at: 12:32:18


### Step 3.3: Verify the Endpoint

Let us verify that the endpoint is in service and get its details.

**Expected output:** Endpoint status should be `InService`.

In [15]:
import boto3

# Create a SageMaker client to check endpoint status
sm_client = boto3.client("sagemaker", region_name=region)

# Describe the endpoint
response = sm_client.describe_endpoint(EndpointName=endpoint_name)

print(f"Endpoint Name:    {response['EndpointName']}")
print(f"Endpoint Status:  {response['EndpointStatus']}")
print(f"Endpoint ARN:     {response['EndpointArn']}")
print(f"Creation Time:    {response['CreationTime']}")
print(f"Last Modified:    {response['LastModifiedTime']}")

if response['EndpointStatus'] == 'InService':
    print("\nThe endpoint is ready to receive inference requests!")
else:
    print(f"\nEndpoint is not yet ready. Current status: {response['EndpointStatus']}")
    print("Please wait a few more minutes and re-run this cell.")

Endpoint Name:    resnet18-endpoint-1775622522
Endpoint Status:  InService
Endpoint ARN:     arn:aws:sagemaker:ap-southeast-1:359143808088:endpoint/resnet18-endpoint-1775622522
Creation Time:    2026-04-08 12:28:48.287000+08:00
Last Modified:    2026-04-08 12:31:58.167000+08:00

The endpoint is ready to receive inference requests!


💡 **Key Concept:** The SageMaker SDK's `deploy()` method abstracts away three separate API calls (CreateModel, CreateEndpointConfig, CreateEndpoint). Behind the scenes, SageMaker pulls the PyTorch container image, provisions an `ml.m5.large` instance, downloads your model from S3, loads it into the container, and starts accepting requests.

---

## 🔧 Exercise 4: Testing the Endpoint (~30 minutes)

Now that our endpoint is live, let us send some test requests and see how it performs.

We will:
1. Send a single inference request
2. Parse and display the results with human-readable class names
3. Send multiple requests and measure response time
4. Test with a real image

### Step 4.1: Send a Single Inference Request

We will create a sample image tensor and send it to the endpoint via JSON.

**Expected output:** Top-5 predicted ImageNet classes with probabilities.

In [16]:
import json
import numpy as np
import torch
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

# Configure the predictor to use JSON serialization
predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

# Create a sample input tensor
# In a real application, this would be a preprocessed image
sample_input = torch.randn(1, 3, 224, 224).tolist()

# Send the request to the endpoint
print("Sending inference request...")
response = predictor.predict({"inputs": sample_input})

# Display the response
print("\nResponse from endpoint:")
print(json.dumps(response, indent=2))

Sending inference request...



Response from endpoint:
{
  "predictions": [
    {
      "class_index": 107,
      "probability": 0.1755399852991104
    },
    {
      "class_index": 611,
      "probability": 0.041730739176273346
    },
    {
      "class_index": 845,
      "probability": 0.029719529673457146
    },
    {
      "class_index": 4,
      "probability": 0.02747942879796028
    },
    {
      "class_index": 111,
      "probability": 0.0207830723375082
    }
  ]
}


### Step 4.2: Map Class Indices to Human-Readable Names

The model outputs class indices (0-999). Let us map these to actual ImageNet class names.

**Expected output:** Class names like "golden retriever", "tabby cat", etc.

In [17]:
import urllib.request
import json

# Download the ImageNet class labels
labels_url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"

try:
    with urllib.request.urlopen(labels_url) as url:
        imagenet_labels = json.loads(url.read().decode())
    print(f"Loaded {len(imagenet_labels)} ImageNet class labels")
except Exception as e:
    print(f"Could not download labels: {e}")
    # Fallback: use class indices
    imagenet_labels = [f"class_{i}" for i in range(1000)]

# Parse the prediction response and display human-readable results
# The response format depends on our inference.py output_fn
if isinstance(response, str):
    result = json.loads(response)
else:
    result = response

print("\nTop 5 Predictions:")
print("-" * 50)
for i, pred in enumerate(result["predictions"], 1):
    class_idx = pred["class_index"]
    probability = pred["probability"]
    class_name = imagenet_labels[class_idx] if class_idx < len(imagenet_labels) else f"class_{class_idx}"
    print(f"  {i}. {class_name:<30} {probability:.4f} ({probability*100:.2f}%)")

Loaded 1000 ImageNet class labels

Top 5 Predictions:
--------------------------------------------------
  1. jellyfish                      0.1755 (17.55%)
  2. jigsaw puzzle                  0.0417 (4.17%)
  3. syringe                        0.0297 (2.97%)
  4. hammerhead shark               0.0275 (2.75%)
  5. nematode                       0.0208 (2.08%)


### Step 4.3: Test with a Real Image

Let us download a sample image from the internet and send it to our endpoint.

**Expected output:** Predictions that make sense for the image content.

In [18]:
import urllib.request
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Download a sample image (a dog)
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg"

try:
    urllib.request.urlretrieve(image_url, "test_image.jpg")
    image = Image.open("test_image.jpg").convert("RGB")
    print(f"Downloaded test image: {image.size[0]}x{image.size[1]}")
    
    # Display the image
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.title("Test Image")
    plt.axis("off")
    plt.show()
    
except Exception as e:
    print(f"Could not download image: {e}")
    print("Creating a synthetic test image instead...")
    image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))

Could not download image: HTTP Error 403: Forbidden
Creating a synthetic test image instead...


In [19]:
# Preprocess the image and send to the endpoint
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Transform and prepare the image
input_tensor = transform(image).unsqueeze(0)  # Add batch dimension
print(f"Input tensor shape: {input_tensor.shape}")

# Send to endpoint
response = predictor.predict({"inputs": input_tensor.tolist()})

# Parse and display results
if isinstance(response, str):
    result = json.loads(response)
else:
    result = response

print("\nPredictions for the test image:")
print("-" * 50)
for i, pred in enumerate(result["predictions"], 1):
    class_idx = pred["class_index"]
    probability = pred["probability"]
    class_name = imagenet_labels[class_idx] if class_idx < len(imagenet_labels) else f"class_{class_idx}"
    bar = "█" * int(probability * 40)
    print(f"  {i}. {class_name:<30} {probability:.4f} {bar}")

Input tensor shape: torch.Size([1, 3, 224, 224])



Predictions for the test image:
--------------------------------------------------
  1. jellyfish                      0.0755 ███
  2. jigsaw puzzle                  0.0368 █
  3. prayer rug                     0.0320 █
  4. doormat                        0.0266 █
  5. hammerhead shark               0.0251 █


### Step 4.4: Measure Endpoint Response Time

Let us send multiple requests and measure the latency. This helps us understand the endpoint's performance characteristics.

**Expected output:** Average, minimum, and maximum response times.

In [20]:
import time

# Send multiple requests and measure response time
num_requests = 10
latencies = []

# Prepare input data once
test_input = {"inputs": torch.randn(1, 3, 224, 224).tolist()}

print(f"Sending {num_requests} inference requests...")
print("-" * 50)

for i in range(num_requests):
    start_time = time.time()
    response = predictor.predict(test_input)
    elapsed = time.time() - start_time
    latencies.append(elapsed)
    print(f"  Request {i+1:2d}: {elapsed*1000:.0f} ms")

# Calculate statistics
avg_latency = sum(latencies) / len(latencies)
min_latency = min(latencies)
max_latency = max(latencies)

print("\nLatency Summary:")
print("-" * 50)
print(f"  Average: {avg_latency*1000:.0f} ms")
print(f"  Min:     {min_latency*1000:.0f} ms")
print(f"  Max:     {max_latency*1000:.0f} ms")
print(f"  Std Dev: {(sum((l - avg_latency)**2 for l in latencies) / len(latencies))**0.5 * 1000:.0f} ms")

Sending 10 inference requests...
--------------------------------------------------


  Request  1: 307 ms


  Request  2: 295 ms


  Request  3: 472 ms


  Request  4: 279 ms


...

  Request  6: 292 ms


  Request  7: 351 ms


  Request  8: 269 ms


  Request  9: 402 ms


  Request 10: 304 ms

Latency Summary:
--------------------------------------------------
  Average: 326 ms
  Min:     269 ms
  Max:     472 ms
  Std Dev: 61 ms


📝 **Note:** The first request may be slower due to model loading (a "warm-up" effect). Subsequent requests should be more consistent. Network latency between your machine and the AWS region also affects response time.

---

## 🔧 Exercise 5: Monitoring with CloudWatch (~30 minutes)

Now that we have a running endpoint, let us look at how to monitor it using Amazon CloudWatch.

We will:
1. Access CloudWatch logs from the endpoint
2. Query invocation metrics (latency, error rate, invocation count)
3. Discuss what to monitor and why
4. Create a simple CloudWatch alarm

💡 **Key Concept:** Monitoring is not optional — it is essential. Without monitoring, you would not know if your model is returning errors, running slowly, or receiving unexpected inputs.

### Step 5.1: Access CloudWatch Logs

SageMaker automatically sends container logs to CloudWatch. These include:
- Container startup logs
- Model loading logs
- Inference request logs (if your code logs them)
- Error messages and stack traces

**Expected output:** Recent log entries from the endpoint's container.

In [21]:
import boto3
from datetime import datetime, timedelta

# Create a CloudWatch Logs client
logs_client = boto3.client("logs", region_name=region)

# SageMaker endpoint logs are stored in this log group pattern:
# /aws/sagemaker/Endpoints/<endpoint-name>
log_group_name = f"/aws/sagemaker/Endpoints/{endpoint_name}"

print(f"Log group: {log_group_name}")
print("=" * 60)

try:
    # Get the log streams (one per container instance)
    log_streams = logs_client.describe_log_streams(
        logGroupName=log_group_name,
        orderBy="LastEventTime",
        descending=True,
        limit=5
    )
    
    if log_streams["logStreams"]:
        # Read the most recent log stream
        latest_stream = log_streams["logStreams"][0]["logStreamName"]
        print(f"Latest log stream: {latest_stream}")
        print("-" * 60)
        
        # Get log events from the past hour
        events = logs_client.get_log_events(
            logGroupName=log_group_name,
            logStreamName=latest_stream,
            startFromHead=False,
            limit=20  # Get the 20 most recent events
        )
        
        for event in events["events"]:
            timestamp = datetime.fromtimestamp(event["timestamp"] / 1000)
            message = event["message"].strip()
            print(f"[{timestamp.strftime('%H:%M:%S')}] {message}")
    else:
        print("No log streams found yet. Logs may take a few minutes to appear.")
        
except logs_client.exceptions.ResourceNotFoundException:
    print(f"Log group {log_group_name} not found.")
    print("Logs may take a few minutes to appear after the endpoint starts.")
except Exception as e:
    print(f"Error accessing logs: {e}")

Log group: /aws/sagemaker/Endpoints/resnet18-endpoint-1775622522
Latest log stream: AllTraffic/i-0002e404a20daba79
------------------------------------------------------------
[12:32:22] 2026-04-08T04:32:22,180 [INFO ] W-9000-model_1.0-stdout org.pytorch.serve.wlm.WorkerLifeCycle - result=[METRICS]PredictionTime.Milliseconds:137.87|#ModelName:model,Level:Model|#type:GAUGE|#hostname:container-0.local,1775622742,f5747622-ef35-4f20-9f45-e2a24f76b053, pattern=[METRICS]
[12:32:22] 2026-04-08T04:32:22,181 [INFO ] W-9000-model_1.0 ACCESS_LOG - /169.254.178.2:46498 "POST /invocations HTTP/1.1" 200 152
[12:32:22] 2026-04-08T04:32:22,181 [INFO ] W-9000-model_1.0 TS_METRICS - Requests2XX.Count:1.0|#Level:Host|#hostname:container-0.local,timestamp:1775622742
[12:32:22] 2026-04-08T04:32:22,182 [INFO ] W-9000-model_1.0 TS_METRICS - ts_inference_latency_microseconds.Microseconds:145820.985|#model_name:model,model_version:default|#hostname:container-0.local,timestamp:1775622742
[12:32:22] 2026-04-08T0

### Step 5.2: Query Invocation Metrics

CloudWatch automatically tracks key metrics for SageMaker endpoints. Let us query the most important ones:

| Metric | What It Tells You |
|--------|-------------------|
| `Invocations` | Total number of inference requests |
| `ModelLatency` | Time the model takes to process a request (in microseconds) |
| `OverheadLatency` | Time spent on SageMaker overhead (routing, serialization) |
| `Invocation4XXErrors` | Client errors (bad requests) |
| `Invocation5XXErrors` | Server errors (model crashes, timeouts) |

**Expected output:** Metric values from the requests we sent in Exercise 4.

In [22]:
import boto3
from datetime import datetime, timedelta

# Create a CloudWatch client
cw_client = boto3.client("cloudwatch", region_name=region)

# Define the time range to query (past 1 hour)
end_time = datetime.utcnow()
start_time = end_time - timedelta(hours=1)

# Metrics to query
metrics_to_query = [
    {"name": "Invocations",          "stat": "Sum",     "unit": "requests"},
    {"name": "ModelLatency",         "stat": "Average", "unit": "microseconds"},
    {"name": "OverheadLatency",      "stat": "Average", "unit": "microseconds"},
    {"name": "Invocation4XXErrors",  "stat": "Sum",     "unit": "errors"},
    {"name": "Invocation5XXErrors",  "stat": "Sum",     "unit": "errors"},
]

print(f"CloudWatch Metrics for: {endpoint_name}")
print(f"Time range: {start_time.strftime('%H:%M')} - {end_time.strftime('%H:%M')} UTC")
print("=" * 60)

for metric in metrics_to_query:
    try:
        response = cw_client.get_metric_statistics(
            Namespace="AWS/SageMaker",
            MetricName=metric["name"],
            Dimensions=[
                {
                    "Name": "EndpointName",
                    "Value": endpoint_name
                },
                {
                    "Name": "VariantName",
                    "Value": "AllTraffic"  # Default variant name
                }
            ],
            StartTime=start_time,
            EndTime=end_time,
            Period=300,  # 5-minute intervals
            Statistics=[metric["stat"]]
        )
        
        datapoints = response["Datapoints"]
        if datapoints:
            # Get the most recent datapoint
            latest = sorted(datapoints, key=lambda x: x["Timestamp"])[-1]
            value = latest[metric["stat"]]
            
            # Convert microseconds to milliseconds for readability
            if "Latency" in metric["name"]:
                print(f"  {metric['name']:<25} {value/1000:.1f} ms")
            else:
                print(f"  {metric['name']:<25} {value:.0f} {metric['unit']}")
        else:
            print(f"  {metric['name']:<25} No data yet")
            
    except Exception as e:
        print(f"  {metric['name']:<25} Error: {e}")

print("\n📝 Note: CloudWatch metrics may take 1-2 minutes to appear after requests.")

CloudWatch Metrics for: resnet18-endpoint-1775622522
Time range: 03:32 - 04:32 UTC


  Invocations               0 requests


  ModelLatency              No data yet
  OverheadLatency           No data yet
  Invocation4XXErrors       No data yet
  Invocation5XXErrors       No data yet

📝 Note: CloudWatch metrics may take 1-2 minutes to appear after requests.


/tmp/ipykernel_55287/4013577905.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time = datetime.utcnow()


### Step 5.3: What to Monitor and Alert On

Here is a practical monitoring checklist for production ML endpoints:

| Metric | Alarm Threshold (Example) | Why |
|--------|--------------------------|-----|
| **5XX Errors** | > 0 in 5 minutes | Your model is crashing |
| **4XX Errors** | > 10% of requests | Clients are sending bad data |
| **Model Latency** | > 1000 ms average | Responses are too slow |
| **Invocation Count** | = 0 for 30 minutes | Nobody is using the endpoint (cost waste?) |
| **CPU Utilisation** | > 80% sustained | Need to scale up |
| **Memory Utilisation** | > 90% sustained | Risk of out-of-memory errors |

### Step 5.4: Create a CloudWatch Alarm

Let us create a simple alarm that triggers if the endpoint returns **any 5XX errors** (server-side errors). In a production setting, this alarm could send an email or trigger an automated remediation.

**Expected output:** A CloudWatch alarm that monitors for server errors.

In [23]:
# Create a CloudWatch alarm for 5XX errors

alarm_name = f"{endpoint_name}-5xx-errors"

try:
    cw_client.put_metric_alarm(
        AlarmName=alarm_name,
        AlarmDescription=(
            f"Alarm when endpoint {endpoint_name} returns 5XX errors. "
            "This indicates the model is crashing or encountering server-side errors."
        ),
        MetricName="Invocation5XXErrors",
        Namespace="AWS/SageMaker",
        Statistic="Sum",
        Dimensions=[
            {
                "Name": "EndpointName",
                "Value": endpoint_name
            },
            {
                "Name": "VariantName",
                "Value": "AllTraffic"
            }
        ],
        Period=300,              # Check every 5 minutes
        EvaluationPeriods=1,     # Trigger after 1 evaluation period
        Threshold=1.0,           # If >= 1 error occurs
        ComparisonOperator="GreaterThanOrEqualToThreshold",
        TreatMissingData="notBreaching",  # No data = no alarm
        # In production, you would add an SNS topic for notifications:
        # AlarmActions=["arn:aws:sns:region:account-id:my-topic"],
    )
    
    print(f"CloudWatch Alarm created: {alarm_name}")
    print(f"\nAlarm configuration:")
    print(f"  Metric:     Invocation5XXErrors")
    print(f"  Threshold:  >= 1 error")
    print(f"  Period:     5 minutes")
    print(f"  Action:     None (demo mode)")
    print(f"\n📝 Note: In production, add an SNS topic to AlarmActions")
    print(f"  to receive email/SMS notifications when the alarm triggers.")
    
except Exception as e:
    print(f"Error creating alarm: {e}")

CloudWatch Alarm created: resnet18-endpoint-1775622522-5xx-errors

Alarm configuration:
  Metric:     Invocation5XXErrors
  Threshold:  >= 1 error
  Period:     5 minutes
  Action:     None (demo mode)

📝 Note: In production, add an SNS topic to AlarmActions
  to receive email/SMS notifications when the alarm triggers.


In [24]:
# Check the alarm status

try:
    alarm_response = cw_client.describe_alarms(AlarmNames=[alarm_name])
    
    if alarm_response["MetricAlarms"]:
        alarm = alarm_response["MetricAlarms"][0]
        print(f"Alarm Name:   {alarm['AlarmName']}")
        print(f"State:        {alarm['StateValue']}")
        print(f"State Reason: {alarm['StateReason']}")
    else:
        print("Alarm not found.")
        
except Exception as e:
    print(f"Error checking alarm: {e}")

Alarm Name:   resnet18-endpoint-1775622522-5xx-errors
State:        INSUFFICIENT_DATA
State Reason: Unchecked: Initial alarm creation


💡 **Key Concept:** CloudWatch alarms let you react to problems automatically. In production, alarms are connected to notification systems (SNS topics) that can email your team, send Slack messages, or even trigger automated rollback procedures.

---

## 🔧 Exercise 6: Cleaning Up (~15 minutes)

⚠️ **Important: This is the most important exercise!**

Cloud resources cost money as long as they are running. An `ml.m5.large` endpoint costs approximately $0.12/hour — that is nearly **$86/month** if left running.

We need to delete:
1. The SageMaker **Endpoint** (the running compute resources)
2. The SageMaker **Endpoint Configuration**
3. The SageMaker **Model**
4. The CloudWatch **Alarm**
5. The S3 **model artifacts** (optional — S3 storage is very cheap)

### Step 6.1: Delete the SageMaker Endpoint

This is the most important step — the endpoint is what costs money.

**Expected output:** Confirmation that the endpoint has been deleted.

In [25]:
import boto3

sm_client = boto3.client("sagemaker", region_name=region)

# Step 1: Delete the endpoint (this stops the compute instances)
print("Step 1: Deleting SageMaker Endpoint...")
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f"  Endpoint '{endpoint_name}' deletion initiated.")
    print(f"  (Full deletion may take 1-2 minutes)")
except sm_client.exceptions.ClientError as e:
    print(f"  Error: {e}")

Step 1: Deleting SageMaker Endpoint...


  Endpoint 'resnet18-endpoint-1775622522' deletion initiated.
  (Full deletion may take 1-2 minutes)


In [26]:
# Step 2: Delete the endpoint configuration
print("Step 2: Deleting Endpoint Configuration...")
try:
    # The endpoint config name usually matches the endpoint name
    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print(f"  Endpoint config '{endpoint_name}' deleted.")
except sm_client.exceptions.ClientError as e:
    # If the config name is different, list and find it
    print(f"  Note: {e}")
    print(f"  The endpoint config may have a different name.")
    print(f"  Check the SageMaker console for orphaned configs.")

Step 2: Deleting Endpoint Configuration...


  Endpoint config 'resnet18-endpoint-1775622522' deleted.


In [27]:
# Step 3: Delete the SageMaker model
print("Step 3: Deleting SageMaker Model...")
try:
    sm_client.delete_model(ModelName="resnet18-image-classifier")
    print(f"  Model 'resnet18-image-classifier' deleted.")
except sm_client.exceptions.ClientError as e:
    print(f"  Note: {e}")

Step 3: Deleting SageMaker Model...


  Model 'resnet18-image-classifier' deleted.


In [28]:
# Step 4: Delete the CloudWatch alarm
print("Step 4: Deleting CloudWatch Alarm...")
try:
    cw_client = boto3.client("cloudwatch", region_name=region)
    cw_client.delete_alarms(AlarmNames=[alarm_name])
    print(f"  Alarm '{alarm_name}' deleted.")
except Exception as e:
    print(f"  Error: {e}")

Step 4: Deleting CloudWatch Alarm...


  Alarm 'resnet18-endpoint-1775622522-5xx-errors' deleted.


In [29]:
# Step 5: Delete S3 model artifacts (optional)
print("Step 5: Deleting S3 model artifacts...")
try:
    s3_client = boto3.client("s3", region_name=region)
    s3_client.delete_object(Bucket=bucket_name, Key=s3_key)
    print(f"  Deleted s3://{bucket_name}/{s3_key}")
except Exception as e:
    print(f"  Error: {e}")

print("\n" + "=" * 60)
print("CLEANUP COMPLETE")
print("=" * 60)
print("\nAll resources have been deleted. You should not incur")
print("any further charges from this session.")
print("\n⚠️  Double-check the SageMaker console to confirm:")
print(f"   https://{region}.console.aws.amazon.com/sagemaker/home?region={region}#/endpoints")

Step 5: Deleting S3 model artifacts...


  Deleted s3://sagemaker-ap-southeast-1-359143808088/models/resnet18/model.tar.gz

CLEANUP COMPLETE

All resources have been deleted. You should not incur
any further charges from this session.

⚠️  Double-check the SageMaker console to confirm:
   https://ap-southeast-1.console.aws.amazon.com/sagemaker/home?region=ap-southeast-1#/endpoints


### Step 6.2: Clean Up Local Files

Let us also remove the local files we created during the exercises.

In [30]:
import shutil
import os

# Clean up local files
local_files_to_delete = [
    "model.tar.gz",
    "test_image.jpg",
]

local_dirs_to_delete = [
    "model_artifacts",
    "docker_example",
]

print("Cleaning up local files...")
for f in local_files_to_delete:
    if os.path.exists(f):
        os.remove(f)
        print(f"  Deleted: {f}")

for d in local_dirs_to_delete:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"  Deleted: {d}/")

print("\nLocal cleanup complete!")

Cleaning up local files...
  Deleted: model.tar.gz
  Deleted: model_artifacts/
  Deleted: docker_example/

Local cleanup complete!


---

## Recap and Key Takeaways

Congratulations! In this session, you have completed the full workflow of deploying a deep learning model to the cloud.

### What We Covered

| # | Topic | Key Takeaway |
|---|-------|-------------|
| 1 | **Why Cloud** | Cloud provides scalability, availability, and managed infrastructure — but watch the costs |
| 2 | **Docker** | Containers package your model + code + dependencies into a portable, reproducible unit |
| 3 | **AWS Ecosystem** | SageMaker (hosting) + S3 (storage) + ECR (containers) + CloudWatch (monitoring) |
| 4 | **SageMaker Deployment** | Upload model to S3 → Create Model → Create Endpoint Config → Deploy Endpoint |
| 5 | **Testing** | Send inference requests via the SageMaker SDK, measure latency, verify predictions |
| 6 | **Monitoring** | CloudWatch tracks invocations, latency, errors — set up alarms for critical metrics |
| 7 | **Cleanup** | Always delete endpoints and resources when done — cloud costs never sleep! |

### The Deployment Workflow (Summary)

```
┌──────────┐    ┌──────────┐    ┌────────────┐    ┌──────────┐    ┌──────────┐
│ 1. Train │    │ 2. Save  │    │ 3. Package │    │ 4. Deploy│    │5. Monitor│
│   Model  │───▶│  & Export │───▶│  & Upload  │───▶│   to     │───▶│  & Manage│
│          │    │          │    │  to S3     │    │SageMaker │    │          │
└──────────┘    └──────────┘    └────────────┘    └──────────┘    └──────────┘
```

### Cost Awareness Reminders

⚠️ **Before you leave, verify:**
1. Your SageMaker endpoint is **deleted** (check the console)
2. No other SageMaker resources are running
3. Set up **AWS Billing Alerts** on your account:
   - Go to AWS Billing → Budgets → Create a budget
   - Set a monthly budget of $10 and receive an alert at 80%

### What is Next: Session 4

In **Session 4**, we will cover **on-premise deployment** as an alternative to cloud deployment:
- When and why to deploy on your own servers
- Using Docker and Docker Compose for local deployment
- Setting up NVIDIA GPU support for on-premise inference
- Comparison of cloud vs on-premise trade-offs

---

### Additional Resources

- [Amazon SageMaker Documentation](https://docs.aws.amazon.com/sagemaker/)
- [SageMaker Python SDK](https://sagemaker.readthedocs.io/)
- [Docker Documentation](https://docs.docker.com/get-started/)
- [AWS Free Tier](https://aws.amazon.com/free/) — Check what services are included
- [SageMaker Pricing](https://aws.amazon.com/sagemaker/pricing/)
- [CloudWatch User Guide](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/)
- [MLOps on AWS](https://aws.amazon.com/sagemaker/mlops/)